# 01 — Extract & Transcribe

Takes an m4b audiobook file, extracts chapters, transcribes selected chapters
using Whisper, and saves transcript fixtures as JSON.

**Workflow:** m4b → ffprobe chapters → ffmpeg WAV slices → Whisper → JSON fixtures

In [5]:
# Parameters (papermill-compatible)
m4b_path = "/Users/eladkatz/Downloads/harry-potter-sorcerers-stone/harry-potter-sorcerers-stone.m4b"
book_slug = "hp-book1-ss"  # Short name for fixture directory
chapters_to_transcribe = []  # empty array - all chapters
whisper_model_size = "small"  # tiny, base, small, medium, large

In [6]:
import sys, os
# Ensure lib/ is importable regardless of CWD
_eval_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if _eval_root not in sys.path:
    sys.path.insert(0, _eval_root)

from pathlib import Path
from lib.audio import extract_chapters, extract_chapter_audio
from lib.whisper_transcribe import load_model, transcribe_and_save
from lib.transcript import save_chapters

## Step 1: Extract chapters from m4b

In [7]:
assert m4b_path, "Set m4b_path in the parameters cell above"
m4b = Path(m4b_path)
assert m4b.exists(), f"File not found: {m4b}"

chapters = extract_chapters(m4b)
print(f"Found {len(chapters)} chapters:\n")
for ch in chapters:
    mins = ch.duration / 60
    print(f"  [{ch.index:2d}] {ch.title:<40s} {mins:.1f} min")

Found 19 chapters:

  [ 0] Opening Credits                          1.1 min
  [ 1] Chapter One - The Boy Who Lived          31.8 min
  [ 2] Chapter Two - The Vanishing Glass        22.5 min
  [ 3] Chapter Three - The Letters from No One  25.5 min
  [ 4] Chapter Four - The Keeper of the Keys    25.4 min
  [ 5] Chapter Five - Diagon Alley              46.3 min
  [ 6] Chapter Six - The Journey from Platform Nine and Three-Quarters 39.7 min
  [ 7] Chapter Seven - The Sorting Hat          30.8 min
  [ 8] Chapter Eight - The Potions Master       19.4 min
  [ 9] Chapter Nine - The Midnight Duel         31.4 min
  [10] Chapter Ten - Halloween                  26.8 min
  [11] Chapter Eleven - Quidditch               21.0 min
  [12] Chapter Twelve - The Mirror of Erised    35.8 min
  [13] Chapter Thirteen - Nicolas Flamel        19.6 min
  [14] Chapter Fourteen - Norbert the Norwegian Ridgeback 21.4 min
  [15] Chapter Fifteen - The Forbidden Forest   33.5 min
  [16] Chapter Sixteen - Through the

In [8]:
# Save chapter metadata
eval_root = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("..").resolve()
fixture_dir = eval_root / "fixtures" / book_slug
fixture_dir.mkdir(parents=True, exist_ok=True)
save_chapters(chapters, fixture_dir / "chapters.json")
print(f"Saved chapter metadata to {fixture_dir / 'chapters.json'}")

Saved chapter metadata to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/chapters.json


## Step 2: Select chapters to transcribe

In [9]:
if chapters_to_transcribe:
    selected = [ch for ch in chapters if ch.index in chapters_to_transcribe]
else:
    selected = chapters

print(f"Will transcribe {len(selected)} chapter(s):")
for ch in selected:
    print(f"  [{ch.index}] {ch.title} ({ch.duration/60:.1f} min)")

Will transcribe 19 chapter(s):
  [0] Opening Credits (1.1 min)
  [1] Chapter One - The Boy Who Lived (31.8 min)
  [2] Chapter Two - The Vanishing Glass (22.5 min)
  [3] Chapter Three - The Letters from No One (25.5 min)
  [4] Chapter Four - The Keeper of the Keys (25.4 min)
  [5] Chapter Five - Diagon Alley (46.3 min)
  [6] Chapter Six - The Journey from Platform Nine and Three-Quarters (39.7 min)
  [7] Chapter Seven - The Sorting Hat (30.8 min)
  [8] Chapter Eight - The Potions Master (19.4 min)
  [9] Chapter Nine - The Midnight Duel (31.4 min)
  [10] Chapter Ten - Halloween (26.8 min)
  [11] Chapter Eleven - Quidditch (21.0 min)
  [12] Chapter Twelve - The Mirror of Erised (35.8 min)
  [13] Chapter Thirteen - Nicolas Flamel (19.6 min)
  [14] Chapter Fourteen - Norbert the Norwegian Ridgeback (21.4 min)
  [15] Chapter Fifteen - The Forbidden Forest (33.5 min)
  [16] Chapter Sixteen - Through the Trapdoor (39.0 min)
  [17] Chapter Seventeen - The Man with Two Faces (37.5 min)
  [18] En

## Step 3: Load Whisper model

In [10]:
print(f"Loading Whisper model: {whisper_model_size}")
model = load_model(whisper_model_size)
print("Model loaded.")

Loading Whisper model: small
Model loaded.


## Step 4: Extract audio & transcribe each chapter

In [11]:
audio_dir = fixture_dir / "audio"
audio_dir.mkdir(exist_ok=True)

for ch in selected:
    print(f"\n--- Chapter {ch.index}: {ch.title} ---")
    
    # Extract audio
    print(f"  Extracting audio ({ch.duration/60:.1f} min)...")
    wav_path = extract_chapter_audio(m4b, ch, audio_dir)
    print(f"  Audio saved to {wav_path}")
    
    # Transcribe
    fixture_path = fixture_dir / f"transcript_ch{ch.index:02d}.json"
    print(f"  Transcribing with Whisper...")
    sentences = transcribe_and_save(
        wav_path,
        fixture_path,
        model=model,
        chapter_offset=ch.start_time,  # Absolute book timestamps
    )
    print(f"  Got {len(sentences)} sentences, saved to {fixture_path}")
    
    # Preview first few sentences
    for s in sentences[:3]:
        mins, secs = divmod(int(s.start_time), 60)
        print(f"    [{mins:02d}:{secs:02d}] {s.text[:80]}")
    if len(sentences) > 3:
        print(f"    ... and {len(sentences) - 3} more")


--- Chapter 0: Opening Credits ---
  Extracting audio (1.1 min)...
  Audio saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/audio/ch00.wav
  Transcribing with Whisper...


/Users/eladkatz/Dev/AudioBook Player/eval/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 6353/6353 [00:04<00:00, 1324.88frames/s]


  Got 3 sentences, saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/transcript_ch00.json
    [00:00] This is Audible.
    [00:11] Potamor Publishing and Audible Studios present Harry Potter, the full cast audio
    [00:58] Audible Studios Audible Studios

--- Chapter 1: Chapter One - The Boy Who Lived ---
  Extracting audio (31.8 min)...
  Audio saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/audio/ch01.wav
  Transcribing with Whisper...


/Users/eladkatz/Dev/AudioBook Player/eval/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
 99%|█████████▉| 188956/190700 [02:44<00:01, 1151.99frames/s]


  Got 391 sentences, saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/transcript_ch01.json
    [01:03] Chapter 1 The Boy Who Lived Mr and Mrs Dursley, of number 4, Privet Drive, were 
    [01:21] Thank you very much.
    [01:22] They were the last people you'd expect to be involved in anything strange or mys
    ... and 388 more

--- Chapter 2: Chapter Two - The Vanishing Glass ---
  Extracting audio (22.5 min)...
  Audio saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/audio/ch02.wav
  Transcribing with Whisper...


...

  Got 690 sentences, saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/transcript_ch16.json
    [431:45] Chapter 16. Through the Trap Door In years to come, Harry would never quite reme
    [432:01] Yet the days crept by, and there could be no doubt that Fluffy was still alive a
    [432:09] It was swelteringly hot, especially in the large classroom where they did their 
    ... and 687 more

--- Chapter 17: Chapter Seventeen - The Man with Two Faces ---
  Extracting audio (37.5 min)...
  Audio saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/audio/ch17.wav
  Transcribing with Whisper...


/Users/eladkatz/Dev/AudioBook Player/eval/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 225100/225100 [03:05<00:00, 1211.60frames/s]


  Got 522 sentences, saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/transcript_ch17.json
    [470:42] Chapter 17. The man with two faces. It was Quirrell.
    [470:52] You?
    [470:54] Quirrell smiled. His face wasn't twitching at all.
    ... and 519 more

--- Chapter 18: End Credits ---
  Extracting audio (13.2 min)...
  Audio saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/audio/ch18.wav
  Transcribing with Whisper...


/Users/eladkatz/Dev/AudioBook Player/eval/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
 96%|█████████▌| 76470/79470 [01:01<00:02, 1249.76frames/s]

  Got 90 sentences, saved to /Users/eladkatz/Dev/AudioBook Player/eval/fixtures/hp-book1-ss/transcript_ch18.json
    [508:13] You have been listening to Harry Potter and the Sorcerer's Stone, written by J.K
    [510:10] And featuring David Ahmad as Wizard in Street 2, Tom Alexander as Ronan, Philip 
    [512:33] Additional Voices By Brianna Squirrel, Harry Tuffin, Joe Troy, Harriet Turnbull,
    ... and 87 more


## Done!

Transcript fixtures are saved in `fixtures/<book-slug>/`. 
Open **02_explore_transcript.ipynb** to inspect them, or jump to **03_prompt_playground.ipynb** to start testing prompts.